In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from IPython.display import display, clear_output

import dataset
import dataset_misc1d
import space
from gp import gp
from gp import creator as gp_creator
from gp import evaluator as gp_evaluator, selector as gp_selector
from gp import crossover as gp_crossover, mutator as gp_mutator
from symbols import syntax_tree
import randstate

In [ ]:
SAMPLE_SIZE = 150
TRAIN_SIZE  = 0.7
NOISE       = 0.
MESH_SIZE   = 100
TEST_MESH_SIZE   = 200

POPSIZE          = 1000
MAX_STREE_DEPTH  = 8
MAX_STREE_LENGTH = 20
GENERATIONS      = 50
GROUP_SIZE       = 5  # tournament selector.
MUTATION_RATE    = 0.15
ELITISM          = 1

RANDSTATE = 1234

In [ ]:
randstate.setstate(RANDSTATE)

S = dataset_misc1d.MagmanDataset()

#S.sample(size=SAMPLE_SIZE, noise=NOISE, mesh=False)
S.load('../data/magman.csv')

S.split(train_size=TRAIN_SIZE)
S.get_plotter().plot(width=8, height=6, plot_knowldege=False)

S_train = dataset.NumpyDataset(S)
S_test  = dataset.NumpyDataset(S, test=True)

In [ ]:
np.seterr(all='ignore')

syntax_tree.SyntaxTreeInfo.set_problem(S_train)

solutionCreator = gp_creator.PTC2RandomSolutionCreator(nvars=S.nvars)

multiMutator = gp_mutator.MultiMutator(
      gp_mutator.SubtreeReplacerMutator(MAX_STREE_DEPTH, MAX_STREE_LENGTH, solutionCreator),
      gp_mutator.FunctionSymbolMutator(),
      gp_mutator.NumericParameterMutator(all=True),
      gp_mutator.NumericParameterMutator(all=False)
      )

linscaler = gp_evaluator.LinearScaler(S_train.y)
evaluator = gp_evaluator.MSEEvaluator(S_train, linscaler=linscaler)

selector  = gp_selector.TournamentSelector(GROUP_SIZE)
crossover = gp_crossover.SubTreeCrossover(MAX_STREE_DEPTH, MAX_STREE_LENGTH)

settings = gp.GPSettings(
      POPSIZE, GENERATIONS, MAX_STREE_DEPTH, MAX_STREE_LENGTH, S_train, S_test,
      creator=solutionCreator,
      evaluator=evaluator,
      selector=selector,
      crossover=crossover,
      mutator=multiMutator,
      corrector=None,
      mutrate=MUTATION_RATE,
      elitism=ELITISM,
      knowledge=S.knowledge)
symb_regressor = gp.SynthfitGP(settings)


fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)

with tqdm(total=symb_regressor.ngen-1) as pbar:
      def on_newgen(genidx, status):
            
            pbar.update(1)
            pbar.set_description(status)

            ax.cla()

            x = np.linspace(S.xl, S.xu, 100)
            ax.plot(x, S.func(x), linestyle='solid', linewidth=2, color='black', label='Ref. Model')

            n_pts = symb_regressor.synth_y_acc.size if symb_regressor.synth_y_acc is not None else 0
            rgba_colors = np.zeros((n_pts,4))
            rgba_colors[:,0] = 1.0
            rgba_colors[:, 3] = symb_regressor.synth_y_acc

            ax.scatter(symb_regressor.synth_X, symb_regressor.synth_y, marker='o', color=rgba_colors, s=2)

            ax.set_xlim(S.xl, S.xu)
            ax.set_ylim(S.yl, S.yu)

            display(fig)
            display(pbar.container)
            clear_output(wait=True)
            
      best_stree, best_eval = symb_regressor.evolve(newgen_callback=on_newgen)
      best_stree = best_eval.scaling.scale_stree(best_stree)

In [ ]:
test_data_evaluator = gp_evaluator.NMSEEvaluator(S_test)
best_stree.clear_output()
print("\n--- Best syntax tree ---")
print(best_stree)
print(best_eval)
print(f"Max depth: {best_stree.get_max_depth()}")
print(f"Length: {best_stree.get_nnodes()}")
print(f"Test NMSE: {test_data_evaluator.evaluate(best_stree).value}")

In [ ]:
best_stree.clear_output()
S.get_plotter().plot(width=8, height=6, plot_knowldege=False, model=best_stree, zoomout=1)
print(S.evaluate_extra(best_stree))

test_mesh           = space.MeshSpace(S_train, S.knowledge, TEST_MESH_SIZE)
test_know_evaluator = gp_evaluator.KnowledgeEvaluator(S.knowledge, test_mesh)

In [ ]:
best_stree.clear_output()
best_stree_scaled = linscaler.scale_stree(best_stree, best_stree(S_train.X))
best_stree_scaled.clear_output()
#S.get_plotter().plot(width=8, height=6, plot_knowldege=False, model=best_stree_scaled, zoomout=1)

best_stree_scaled.clear_output()
print(f"Test NMSE: {test_data_evaluator.evaluate(best_stree_scaled).value}")

In [ ]:
symb_regressor.stats.plot()